# Práctica 3: SISTEMAS BASADOS EN REGLAS

## 1. Inicialización del entorno CLIPS

In [1]:
#!pip install clipspy
import sys
!{sys.executable} -m pip install clipspy


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#Inicialización del entorno
import clips
import logging

     
def print_rules():
    """
    Inspecciona y muestra en pantalla todas las reglas (defrules) que han sido 
    cargadas y compilaron con éxito en el entorno de CLIPS.
    """
    for r in env.rules():
        print(r)
        
def print_agenda():
    """
    Muestra el estado actual de la Agenda de CLIPS.
    La Agenda contiene las 'activaciones', es decir, aquellas reglas cuyas condiciones 
    (lado izquierdo / LHS) se han satisfecho con los hechos actuales de la memoria de trabajo 
    y que están a la espera de ser ejecutadas al llamar a env.run().
    """
    # env.activations() devuelve la lista de reglas listas para dispararse (junto con los hechos que las activaron)
    for a in env.activations():
        print(a)
        
def print_facts():
    """
    Imprime todos los hechos (facts) almacenados actualmente en la memoria de trabajo (Working Memory).
    Para cada hecho, muestra su índice numérico asignado por CLIPS (f.index) y su contenido/estructura.
    """
    for f in env.facts():
        print(f.index, f)

def crear_entorno():
    """
    Inicializa un nuevo entorno de trabajo independiente en CLIPS, configura el sistema
    de logs de Python y redirige las salidas por pantalla internas de CLIPS (salidas estándar y de error)
    hacia el logger de Python.
    
    Retorna:
        env (clips.Environment): El objeto del entorno de CLIPS listo para ser usado.
    """
    env = clips.Environment()
    logging.basicConfig(level = 10, format = '%(message)s')
    router = clips.LoggingRouter()
    env.add_router(router)
    return env

In [3]:
env = crear_entorno()

env.define_function(print_facts)
env.define_function(print_agenda)
env.define_function(print_rules)

## 2. Descripción completa del problema a resolver

Un robot tiene que suministrar bebidas a las peticiones que se hacen de **6 puestos diferentes**. Para ello el robot utiliza un carro que es parte inseparable de su anatomía y que tiene una capacidad para **3 botellas**. 

Existe una vía que comunica consecutivamente los puestos 1,2,3,4,5 y 6 y que se extiende, además, del puesto 6 al almacén y del almacén al puesto 1, tal y como se indica en la figura. 

        Almacén ----> 1 ----> 2 ----> 3 
            |                         |
            6 <----------5 <--------- 4

**Especificaciones de operación**:

1. **Desplazamiento**: El robot tiene que desplazarse físicamente hasta el puesto donde va a atender la solicitud; sólo puede desplazarse en el sentido que indican las flechas y avanzando un solo puesto en cada acción de movimiento.
2. **Reabastecimiento**: En el almacén el robot puede llenar el carro de bebidas; supondremos que el robot siempre repone existencias hasta completar su capacidad de 3 botellas.
3. **Condiciones del pedido**: El número de botellas que solicita cada puesto nunca será mayor que 3 y el robot debe suministrar el total de botellas solicitado por un puesto en una sola entrega
4. **Estado inicial**: El robot se encuentra en el almacén con el carro de bebidas vacío y desde uno o varios puestos se solicitan un número de botellas.
5. **EStado final/objetivo**: La situación final ha de ser que todas las peticiones de bebidas hayan quedado satisfechas, el robot debe estar de nuevo en el almacén y con el carro de bebidas lleno (3 botellas). 

 Se pide diseñar un sistema basado en reglas con la sintaxis de CLIPS.

## Paso 1: Diseñar la BH: ¿Qué hechos vamos a necesitar?
Identificar y definir los hechos necesarios en CLIPS (utilizando bloques deffacts) para representar la memoria de trabajo:

1. Dónde está el robot.
2. Cómo se comunican los puestos y el almacén (conectividad del mapa).
3. Cuántas botellas lleva el robot en el carro.
4. Qué pedidos/solicitudes hay pendientes en los puestos.
5. Hechos auxiliares o de control necesarios.

### **Para describir los hechos:**

<pre style="background-color: #f5f5f5; border: 1px solid #ccc; padding: 10px; border-radius: 4px;color:#000000; width: 500px; height: 400px; overflow: auto;">
init = 
"""
(deffacts inicio
    ( ... )
    ( ... )
)
"""

Por ejemplo:
    
init = """
(deffacts inicio 
    (carretera ciuA ciuB)
    (localizacion pepe ciuA)
)
"""
</pre>
### 
**Para compilar los hechos del comando `deffacts`, ejecutar** `env.build(init)`

**Para ejecutar el motor de inferencia (cargar hechos en la BH e instancias de reglas en la agenda), ejecutar** `env.reset()`

**Para ver el estado de la BH, ejecutar** `print_facts()`

### Diseño de la Base de Hechos

| Información | Hecho(s) en CLIPS | Comentario |
|---|---|---|
| 1. Dónde está el robot | `(robot (posicion almacen) ...)` | Plantilla `deftemplate robot`: así una sola instrucción `modify` cambia el estado del robot |
| 3. Botellas en el carro | `(robot ... (botellas 0) ...)` | Campo de la misma plantilla, restringido al rango 0..3 |
| 2. Conectividad del mapa | `(siguiente almacen p1)` ... `(siguiente p6 almacen)` | Un hecho por arco **dirigido** de la vía: el robot solo puede avanzar en el sentido de las flechas, un puesto por acción |
| 4. Pedidos pendientes | `(pedido p2 2)` | Puesto y número de botellas solicitadas (≤ 3). Al servirse se retira y se crea `(servido p2 2)` |
| 5. Hechos auxiliares / control | `(capacidad 3)`, campo `acciones` del robot | `capacidad` evita fijar el 3 dentro de las reglas; `acciones` cuenta las acciones ejecutadas, para seguir la traza |

Usamos la plantilla `robot` en vez de hechos ordenados como `(robot almacen)` y `(carro 0)`. Así posición y carga se cambian a la vez con `modify` y nunca hay dos hechos del robot a la vez.

**Escenario de prueba (estado inicial):** el robot está en el almacén con el carro vacío. Hay pedidos en los puestos 2 (2 botellas), 3 (3 botellas), 5 (1 botella) y 6 (2 botellas).

In [4]:
# Partimos de un entorno limpio (permite re-ejecutar esta celda sin errores de redefinición)
env.clear()

# Plantilla para representar el estado del robot (posición + carga del carro + contador de acciones)
plantilla = """
(deftemplate robot
   (slot posicion (type SYMBOL) (default almacen))
   (slot botellas (type INTEGER) (range 0 3) (default 0))
   (slot acciones (type INTEGER) (default 0))
)
"""

init = """
(deffacts inicio
   ; --- 1 y 3. Estado del robot: en el almacen con el carro vacio ---
   (robot (posicion almacen) (botellas 0) (acciones 0))

   ; --- 5. Hechos auxiliares: capacidad maxima del carro ---
   (capacidad 3)

   ; --- 2. Conectividad (arcos dirigidos de la via) ---
   (siguiente almacen p1)
   (siguiente p1 p2)
   (siguiente p2 p3)
   (siguiente p3 p4)
   (siguiente p4 p5)
   (siguiente p5 p6)
   (siguiente p6 almacen)

   ; --- 4. Pedidos pendientes: (pedido <puesto> <num-botellas>) ---
   (pedido p2 2)
   (pedido p3 3)
   (pedido p5 1)
   (pedido p6 2)
)
"""

env.build(plantilla)
env.build(init)
env.reset()
print_facts()

1 (robot (posicion almacen) (botellas 0) (acciones 0))
2 (capacidad 3)
3 (siguiente almacen p1)
4 (siguiente p1 p2)
5 (siguiente p2 p3)
6 (siguiente p3 p4)
7 (siguiente p4 p5)
8 (siguiente p5 p6)
9 (siguiente p6 almacen)
10 (pedido p2 2)
11 (pedido p3 3)
12 (pedido p5 1)
13 (pedido p6 2)


## Paso 2: Diseñar las reglas: ¿Qué acciones puede realizar el robot?

Definir las reglas de producción en CLIPS que modelan las acciones que puede realizar el robot:

1. Moverse: Avanzar de un puesto/almacén al siguiente en la vía.
2. Cargar: Reponer botellas hasta el máximo (3) únicamente estando en el almacén.
3. Servir: Entregar las botellas en un puesto si el robot está en dicho puesto y dispone de botellas suficientes.

### **Para definir las reglas:**

<pre style="background-color: #f5f5f5; border: 1px solid #ccc; padding: 10px; border-radius: 4px;color:#000000; width: 500px; height: 180px; overflow: auto;">
r1 = """
(defrule regla-mover
  ( ... )
  =>
  ( ... )
)
"""
</pre>

### 
**Para compilar las reglas definidas en los comandos `defrule`, ejecutar** `env.build(r1)`

**Para ejecutar el motor de inferencia (cargar hechos en la BH e instancias de reglas en la agenda), ejecutar** `env.reset()`

**Para ver el estado de la BH, ejecutar** `print_facts()`

**Para ver el estado de la agenda ejecutar** `print_agenda()`

### Diseño de las reglas

| Regla | Prioridad (`salience`) | Condición (LHS) | Acción (RHS) |
|---|---|---|---|
| `cargar` | 20 | Robot en el almacén con menos botellas que la capacidad | Llena el carro hasta 3 |
| `servir` | 20 | Robot en un puesto con un pedido pendiente y botellas suficientes (`botellas >= cantidad`) | Retira el pedido, crea `(servido ...)` y descuenta las botellas |
| `mover` | 0 | Quedan pedidos pendientes (`exists`) | Avanza al siguiente nodo de la vía |
| `volver-almacen` | 0 | No quedan pedidos y el robot no está en el almacén | Avanza al siguiente nodo, camino del almacén |

**Control mediante prioridades:**
- `cargar` y `servir` tienen más prioridad que los movimientos. El robot nunca pasa de largo por un puesto que puede atender ni sale del almacén sin llenar el carro.
- Si el robot está en un puesto pero no lleva botellas suficientes, `servir` no se activa. Como el pedido debe entregarse completo en una sola entrega, el robot sigue avanzando y lo atenderá en la siguiente vuelta, después de recargar.
- Separar `mover` y `volver-almacen` evita que el robot dé vueltas sin fin: cuando ya no quedan pedidos solo puede avanzar hasta llegar al almacén.
- Todas las reglas modifican el hecho `robot` con `modify`. Así se crea un hecho nuevo, las activaciones antiguas desaparecen de la agenda y no se producen disparos repetidos.

**Por qué el sistema siempre termina:** cada vez que el robot pasa por el almacén llena el carro (3 botellas). Como ningún pedido supera las 3 botellas, en cada vuelta sirve al menos el primer pedido pendiente que encuentra. Por tanto, el número de pedidos pendientes baja en cada vuelta hasta llegar a 0.

In [5]:
r1 = """
(defrule cargar
   (declare (salience 20))
   ?r <- (robot (posicion almacen) (botellas ?b) (acciones ?n))
   (capacidad ?max)
   (test (< ?b ?max))
   =>
   (modify ?r (botellas ?max) (acciones (+ ?n 1)))
   (printout t "[" (+ ?n 1) "] CARGAR en almacen: " ?b " -> " ?max " botellas" crlf)
)
"""

r2 = """
(defrule servir
   (declare (salience 20))
   ?r <- (robot (posicion ?p) (botellas ?b) (acciones ?n))
   ?ped <- (pedido ?p ?cant)
   (test (>= ?b ?cant))
   =>
   (retract ?ped)
   (assert (servido ?p ?cant))
   (modify ?r (botellas (- ?b ?cant)) (acciones (+ ?n 1)))
   (printout t "[" (+ ?n 1) "] SERVIR " ?cant " botella(s) en " ?p ". Quedan " (- ?b ?cant) " en el carro" crlf)
)
"""

r3 = """
(defrule mover
   (declare (salience 0))
   ?r <- (robot (posicion ?origen) (botellas ?b) (acciones ?n))
   (siguiente ?origen ?destino)
   (exists (pedido ? ?))
   =>
   (modify ?r (posicion ?destino) (acciones (+ ?n 1)))
   (printout t "[" (+ ?n 1) "] MOVER " ?origen " -> " ?destino " (carro: " ?b ")" crlf)
)
"""

r4 = """
(defrule volver-almacen
   (declare (salience 0))
   ?r <- (robot (posicion ?origen&~almacen) (botellas ?b) (acciones ?n))
   (siguiente ?origen ?destino)
   (not (pedido ? ?))
   =>
   (modify ?r (posicion ?destino) (acciones (+ ?n 1)))
   (printout t "[" (+ ?n 1) "] VOLVER " ?origen " -> " ?destino " (sin pedidos pendientes)" crlf)
)
"""

reglas = [r1, r2, r3, r4]
for r in reglas:
    env.build(r)

env.reset()
print("--- REGLAS ---")
print_rules()
print("\n--- HECHOS ---")
print_facts()
print("\n--- AGENDA ---")
print_agenda()

--- REGLAS ---
(defrule MAIN::cargar (declare (salience 20)) ?r <- (robot (posicion almacen) (botellas ?b) (acciones ?n)) (capacidad ?max) (test (< ?b ?max)) => (modify ?r (botellas ?max) (acciones (+ ?n 1))) (printout t "[" (+ ?n 1) "] CARGAR en almacen: " ?b " -> " ?max " botellas" crlf))
(defrule MAIN::servir (declare (salience 20)) ?r <- (robot (posicion ?p) (botellas ?b) (acciones ?n)) ?ped <- (pedido ?p ?cant) (test (>= ?b ?cant)) => (retract ?ped) (assert (servido ?p ?cant)) (modify ?r (botellas (- ?b ?cant)) (acciones (+ ?n 1))) (printout t "[" (+ ?n 1) "] SERVIR " ?cant " botella(s) en " ?p ". Quedan " (- ?b ?cant) " en el carro" crlf))
(defrule MAIN::mover (declare (salience 0)) ?r <- (robot (posicion ?origen) (botellas ?b) (acciones ?n)) (siguiente ?origen ?destino) (exists (pedido ? ?)) => (modify ?r (posicion ?destino) (acciones (+ ?n 1))) (printout t "[" (+ ?n 1) "] MOVER " ?origen " -> " ?destino " (carro: " ?b ")" crlf))
(defrule MAIN::volver-almacen (declare (salience 

En el estado inicial solo hay dos reglas en la agenda: `cargar` (prioridad 20) y `mover` (prioridad 0). Al tener más prioridad, `cargar` se ejecutará primero.

## Paso 3: Probar las reglas

Para cada regla:
- Preparar un estado en la BH en el que la regla examinada debería aplicarse.
- Ejecutar la regla.
- Observar que el estado resultante coincide con el esperado.

El comando `env.reset()` lanza el motor de inferencia y se realiza el primer pattern-matching, introduciendo en la agenda las reglas instanciadas.

Con el comando `print_facts()` puedes comprobar el estado de la BH; con el comando `print_agenda()` puedes ver las reglas instanciadas en la agenda tras la primera iteración del proceso de pattern-matching. 

Con el comando `env.run(1)`, el motor de inferencia ejecuta la RHS de la primera instancia de la agenda y actualiza la BH según los cambios indicados en la RHS. Nada más terminar la ejecución de la regla, se realiza la siguiente iteración del algoritmo, es decir, se realiza el siguiente proceso de pattern-matching. Ejecutando, de nuevo,  `env.run(1)`, se ejecutará la RHS de la primera instancia de la agenda y se modificará la BH de acuerdo a las instrucciones de la RHS de la instancia de regla. Y así sucesivamente.

In [6]:
def preparar_prueba(hechos_prueba, reglas_prueba):
    """
    Deja el entorno con la plantilla, las reglas indicadas y un estado de prueba
    (bloque deffacts 'prueba'), ejecuta reset y muestra la BH y la agenda.
    """
    env.clear()
    env.build(plantilla)
    env.build("(deffacts prueba " + hechos_prueba + ")")
    for r in reglas_prueba:
        env.build(r)
    env.reset()
    print("--- BH ANTES ---")
    print_facts()
    print("\n--- AGENDA ---")
    print_agenda()

def estado_robot():
    """Devuelve (posicion, botellas) del hecho robot actual."""
    for f in env.facts():
        if f.template.name == 'robot':
            return str(f['posicion']), f['botellas']

def ejecutar_un_paso():
    """Dispara una única regla (la primera de la agenda) y muestra la BH resultante."""
    print("\n>>> env.run(1)")
    env.run(1)
    print("\n--- BH DESPUÉS ---")
    print_facts()

conexiones = """
   (capacidad 3)
   (siguiente almacen p1) (siguiente p1 p2) (siguiente p2 p3) (siguiente p3 p4)
   (siguiente p4 p5) (siguiente p5 p6) (siguiente p6 almacen)
"""

### Prueba 3.1 – Regla `cargar`
**Estado:** robot en el almacén con 1 botella y un pedido pendiente en `p1`.
**Esperado:** en la agenda están `cargar` y `mover`. Se dispara `cargar` (más prioridad) y el carro pasa a tener 3 botellas sin que el robot se mueva.

In [7]:
preparar_prueba(conexiones + """
   (robot (posicion almacen) (botellas 1))
   (pedido p1 1)
""", reglas)
ejecutar_un_paso()
assert estado_robot() == ('almacen', 3)
print("\nOK -> el robot sigue en el almacen y el carro está lleno:", estado_robot())

[1] CARGAR en almacen: 1 -> 3 botellas


--- BH ANTES ---
1 (capacidad 3)
2 (siguiente almacen p1)
3 (siguiente p1 p2)
4 (siguiente p2 p3)
5 (siguiente p3 p4)
6 (siguiente p4 p5)
7 (siguiente p5 p6)
8 (siguiente p6 almacen)
9 (robot (posicion almacen) (botellas 1) (acciones 0))
10 (pedido p1 1)

--- AGENDA ---
20 cargar: f-9,f-1
0 mover: f-9,f-2,*

>>> env.run(1)

--- BH DESPUÉS ---
1 (capacidad 3)
2 (siguiente almacen p1)
3 (siguiente p1 p2)
4 (siguiente p2 p3)
5 (siguiente p3 p4)
6 (siguiente p4 p5)
7 (siguiente p5 p6)
8 (siguiente p6 almacen)
9 (robot (posicion almacen) (botellas 3) (acciones 1))
10 (pedido p1 1)

OK -> el robot sigue en el almacen y el carro está lleno: ('almacen', 3)


### Prueba 3.2 – Regla `servir`
**Estado:** robot en `p3` con 3 botellas y un pedido de 2 botellas en `p3`.
**Esperado:** se dispara `servir`: desaparece `(pedido p3 2)`, aparece `(servido p3 2)` y en el carro queda 1 botella.

In [8]:
preparar_prueba(conexiones + """
   (robot (posicion p3) (botellas 3))
   (pedido p3 2)
""", reglas)
ejecutar_un_paso()
assert estado_robot() == ('p3', 1)
print("\nOK -> pedido servido, botellas restantes:", estado_robot()[1])

[1] SERVIR 2 botella(s) en p3. Quedan 1 en el carro


--- BH ANTES ---
1 (capacidad 3)
2 (siguiente almacen p1)
3 (siguiente p1 p2)
4 (siguiente p2 p3)
5 (siguiente p3 p4)
6 (siguiente p4 p5)
7 (siguiente p5 p6)
8 (siguiente p6 almacen)
9 (robot (posicion p3) (botellas 3) (acciones 0))
10 (pedido p3 2)

--- AGENDA ---
20 servir: f-9,f-10
0 mover: f-9,f-5,*

>>> env.run(1)

--- BH DESPUÉS ---
1 (capacidad 3)
2 (siguiente almacen p1)
3 (siguiente p1 p2)
4 (siguiente p2 p3)
5 (siguiente p3 p4)
6 (siguiente p4 p5)
7 (siguiente p5 p6)
8 (siguiente p6 almacen)
9 (robot (posicion p3) (botellas 1) (acciones 1))
11 (servido p3 2)

OK -> pedido servido, botellas restantes: 1


### Prueba 3.3 – `servir` **no** se aplica si no hay botellas suficientes
**Estado:** robot en `p3` con solo 1 botella y un pedido de 2 botellas en `p3`.
**Esperado:** `servir` no aparece en la agenda, porque el pedido debe entregarse completo de una sola vez. Solo se activa `mover` y el robot avanza a `p4`.

In [9]:
preparar_prueba(conexiones + """
   (robot (posicion p3) (botellas 1))
   (pedido p3 2)
""", reglas)
assert all(a.name != 'servir' for a in env.activations())
ejecutar_un_paso()
assert estado_robot() == ('p4', 1)
print("\nOK -> no se sirve parcialmente; el robot avanza:", estado_robot())

[1] MOVER p3 -> p4 (carro: 1)


--- BH ANTES ---
1 (capacidad 3)
2 (siguiente almacen p1)
3 (siguiente p1 p2)
4 (siguiente p2 p3)
5 (siguiente p3 p4)
6 (siguiente p4 p5)
7 (siguiente p5 p6)
8 (siguiente p6 almacen)
9 (robot (posicion p3) (botellas 1) (acciones 0))
10 (pedido p3 2)

--- AGENDA ---
0 mover: f-9,f-5,*

>>> env.run(1)

--- BH DESPUÉS ---
1 (capacidad 3)
2 (siguiente almacen p1)
3 (siguiente p1 p2)
4 (siguiente p2 p3)
5 (siguiente p3 p4)
6 (siguiente p4 p5)
7 (siguiente p5 p6)
8 (siguiente p6 almacen)
9 (robot (posicion p4) (botellas 1) (acciones 1))
10 (pedido p3 2)

OK -> no se sirve parcialmente; el robot avanza: ('p4', 1)


### Prueba 3.4 – Regla `mover`
**Estado:** robot en `p6` con 0 botellas y un pedido pendiente en `p2`.
**Esperado:** se dispara `mover` y el robot pasa de `p6` al `almacen`, que es el único sucesor de `p6` en la vía.

In [10]:
preparar_prueba(conexiones + """
   (robot (posicion p6) (botellas 0))
   (pedido p2 1)
""", reglas)
ejecutar_un_paso()
assert estado_robot() == ('almacen', 0)
print("\nOK -> el robot respeta el sentido de la vía:", estado_robot())

[1] MOVER p6 -> almacen (carro: 0)


--- BH ANTES ---
1 (capacidad 3)
2 (siguiente almacen p1)
3 (siguiente p1 p2)
4 (siguiente p2 p3)
5 (siguiente p3 p4)
6 (siguiente p4 p5)
7 (siguiente p5 p6)
8 (siguiente p6 almacen)
9 (robot (posicion p6) (botellas 0) (acciones 0))
10 (pedido p2 1)

--- AGENDA ---
0 mover: f-9,f-8,*

>>> env.run(1)

--- BH DESPUÉS ---
1 (capacidad 3)
2 (siguiente almacen p1)
3 (siguiente p1 p2)
4 (siguiente p2 p3)
5 (siguiente p3 p4)
6 (siguiente p4 p5)
7 (siguiente p5 p6)
8 (siguiente p6 almacen)
9 (robot (posicion almacen) (botellas 0) (acciones 1))
10 (pedido p2 1)

OK -> el robot respeta el sentido de la vía: ('almacen', 0)


### Prueba 3.5 – Regla `volver-almacen`
**Estado:** robot en `p4` con 1 botella y sin pedidos pendientes.
**Esperado:** `mover` no se activa porque no quedan pedidos. Se dispara `volver-almacen` y el robot pasa a `p5`. Si ejecutamos hasta el final, el robot llega al almacén y recarga.

In [11]:
preparar_prueba(conexiones + """
   (robot (posicion p4) (botellas 1))
""", reglas)
ejecutar_un_paso()
assert estado_robot() == ('p5', 1)

print("\n>>> env.run() hasta vaciar la agenda")
env.run()
print_facts()
assert estado_robot() == ('almacen', 3)
print("\nOK -> el robot vuelve al almacén y llena el carro:", estado_robot())

[1] VOLVER p4 -> p5 (sin pedidos pendientes)
[2] VOLVER p5 -> p6 (sin pedidos pendientes)
[3] VOLVER p6 -> almacen (sin pedidos pendientes)
[4] CARGAR en almacen: 1 -> 3 botellas


--- BH ANTES ---
1 (capacidad 3)
2 (siguiente almacen p1)
3 (siguiente p1 p2)
4 (siguiente p2 p3)
5 (siguiente p3 p4)
6 (siguiente p4 p5)
7 (siguiente p5 p6)
8 (siguiente p6 almacen)
9 (robot (posicion p4) (botellas 1) (acciones 0))

--- AGENDA ---
0 volver-almacen: f-9,f-6,*

>>> env.run(1)

--- BH DESPUÉS ---
1 (capacidad 3)
2 (siguiente almacen p1)
3 (siguiente p1 p2)
4 (siguiente p2 p3)
5 (siguiente p3 p4)
6 (siguiente p4 p5)
7 (siguiente p5 p6)
8 (siguiente p6 almacen)
9 (robot (posicion p5) (botellas 1) (acciones 1))

>>> env.run() hasta vaciar la agenda
1 (capacidad 3)
2 (siguiente almacen p1)
3 (siguiente p1 p2)
4 (siguiente p2 p3)
5 (siguiente p3 p4)
6 (siguiente p4 p5)
7 (siguiente p5 p6)
8 (siguiente p6 almacen)
9 (robot (posicion almacen) (botellas 3) (acciones 4))

OK -> el robot vuelve al almacén y llena el carro: ('almacen', 3)


## Paso 4: Definir el estado objetivo y ejecutar el problema completamente

- Diseñar una regla final con mayor prioridad (`salience`) donde se refleje la siguiente situación:  todas las peticiones de bebidas hayan quedado satisfechas, 
el robot debe estar de nuevo en el almacén y con el carro de bebidas lleno.
- Probar la ejecución completa del sistema sobre el escenario de prueba inicial utilizando `env.run()` y verificar que el motor de inferencia alcanza es estado final de forma correcta.

<pre style="background-color: #f5f5f5; border: 1px solid #ccc; padding: 10px; border-radius: 4px;color:#000000; width: 650px; height: 250px; overflow: auto;">
robj = """
(defrule objetivo
  (declare (salience 100))
  ( ... )
  =>
  (printout t "Se ha alcanzado el estado final")
  (halt)
)
"""
</pre>

In [12]:
robj = """
(defrule objetivo
   (declare (salience 100))
   (robot (posicion almacen) (botellas 3) (acciones ?n))
   (not (pedido ? ?))
   =>
   (printout t "Se ha alcanzado el estado final (" ?n " acciones)" crlf)
   (halt)
)
"""

# Sistema completo: plantilla + hechos iniciales + reglas de acción + regla objetivo
env.clear()
env.build(plantilla)
env.build(init)
for r in reglas + [robj]:
    env.build(r)

env.reset()
print("--- ESTADO INICIAL ---")
print_facts()
print("\n--- AGENDA INICIAL ---")
print_agenda()
print("\n=== EJECUCIÓN COMPLETA ===")
env.run()

[1] CARGAR en almacen: 0 -> 3 botellas
[2] MOVER almacen -> p1 (carro: 3)
[3] MOVER p1 -> p2 (carro: 3)
[4] SERVIR 2 botella(s) en p2. Quedan 1 en el carro
[5] MOVER p2 -> p3 (carro: 1)
[6] MOVER p3 -> p4 (carro: 1)
[7] MOVER p4 -> p5 (carro: 1)
[8] SERVIR 1 botella(s) en p5. Quedan 0 en el carro
[9] MOVER p5 -> p6 (carro: 0)
[10] MOVER p6 -> almacen (carro: 0)
[11] CARGAR en almacen: 0 -> 3 botellas
[12] MOVER almacen -> p1 (carro: 3)
[13] MOVER p1 -> p2 (carro: 3)
[14] MOVER p2 -> p3 (carro: 3)
[15] SERVIR 3 botella(s) en p3. Quedan 0 en el carro
[16] MOVER p3 -> p4 (carro: 0)
[17] MOVER p4 -> p5 (carro: 0)
[18] MOVER p5 -> p6 (carro: 0)
[19] MOVER p6 -> almacen (carro: 0)
[20] CARGAR en almacen: 0 -> 3 botellas
[21] MOVER almacen -> p1 (carro: 3)
[22] MOVER p1 -> p2 (carro: 3)
[23] MOVER p2 -> p3 (carro: 3)
[24] MOVER p3 -> p4 (carro: 3)
[25] MOVER p4 -> p5 (carro: 3)
[26] MOVER p5 -> p6 (carro: 3)
[27] SERVIR 2 botella(s) en p6. Quedan 1 en el carro
[28] VOLVER p6 -> almacen (sin p

--- ESTADO INICIAL ---
1 (robot (posicion almacen) (botellas 0) (acciones 0))
2 (capacidad 3)
3 (siguiente almacen p1)
4 (siguiente p1 p2)
5 (siguiente p2 p3)
6 (siguiente p3 p4)
7 (siguiente p4 p5)
8 (siguiente p5 p6)
9 (siguiente p6 almacen)
10 (pedido p2 2)
11 (pedido p3 3)
12 (pedido p5 1)
13 (pedido p6 2)

--- AGENDA INICIAL ---
20 cargar: f-1,f-2
0 mover: f-1,f-3,*

=== EJECUCIÓN COMPLETA ===


30

In [13]:
print("--- ESTADO FINAL ---")
print_facts()

pendientes = [f for f in env.facts() if f.template.name == 'pedido']
assert not pendientes, "Quedan pedidos sin servir"
assert estado_robot() == ('almacen', 3)
print("\nOK -> todos los pedidos servidos y el robot está en el almacén con el carro lleno")

--- ESTADO FINAL ---
1 (robot (posicion almacen) (botellas 3) (acciones 29))
2 (capacidad 3)
3 (siguiente almacen p1)
4 (siguiente p1 p2)
5 (siguiente p2 p3)
6 (siguiente p3 p4)
7 (siguiente p4 p5)
8 (siguiente p5 p6)
9 (siguiente p6 almacen)
14 (servido p2 2)
15 (servido p5 1)
16 (servido p3 3)
17 (servido p6 2)

OK -> todos los pedidos servidos y el robot está en el almacén con el carro lleno


### Prueba adicional: los 6 puestos piden bebidas

Escenario más exigente, con pedidos en todos los puestos, para comprobar que el sistema funciona con cualquier estado inicial válido y no solo con el del ejemplo.

In [14]:
init2 = """
(deffacts inicio
   (robot (posicion almacen) (botellas 0) (acciones 0))
   (capacidad 3)
   (siguiente almacen p1) (siguiente p1 p2) (siguiente p2 p3) (siguiente p3 p4)
   (siguiente p4 p5) (siguiente p5 p6) (siguiente p6 almacen)
   (pedido p1 1) (pedido p2 3) (pedido p3 2) (pedido p4 1) (pedido p5 3) (pedido p6 2)
)
"""
env.clear()
env.build(plantilla)
env.build(init2)
for r in reglas + [robj]:
    env.build(r)
env.reset()
env.run()

print("\n--- ESTADO FINAL ---")
print_facts()
assert not [f for f in env.facts() if f.template.name == 'pedido']
assert estado_robot() == ('almacen', 3)

[1] CARGAR en almacen: 0 -> 3 botellas
[2] MOVER almacen -> p1 (carro: 3)
[3] SERVIR 1 botella(s) en p1. Quedan 2 en el carro
[4] MOVER p1 -> p2 (carro: 2)
[5] MOVER p2 -> p3 (carro: 2)
[6] SERVIR 2 botella(s) en p3. Quedan 0 en el carro
[7] MOVER p3 -> p4 (carro: 0)
[8] MOVER p4 -> p5 (carro: 0)
[9] MOVER p5 -> p6 (carro: 0)
[10] MOVER p6 -> almacen (carro: 0)
[11] CARGAR en almacen: 0 -> 3 botellas
[12] MOVER almacen -> p1 (carro: 3)
[13] MOVER p1 -> p2 (carro: 3)
[14] SERVIR 3 botella(s) en p2. Quedan 0 en el carro
[15] MOVER p2 -> p3 (carro: 0)
[16] MOVER p3 -> p4 (carro: 0)
[17] MOVER p4 -> p5 (carro: 0)
[18] MOVER p5 -> p6 (carro: 0)
[19] MOVER p6 -> almacen (carro: 0)
[20] CARGAR en almacen: 0 -> 3 botellas
[21] MOVER almacen -> p1 (carro: 3)
[22] MOVER p1 -> p2 (carro: 3)
[23] MOVER p2 -> p3 (carro: 3)
[24] MOVER p3 -> p4 (carro: 3)
[25] SERVIR 1 botella(s) en p4. Quedan 2 en el carro
[26] MOVER p4 -> p5 (carro: 2)
[27] MOVER p5 -> p6 (carro: 2)
[28] SERVIR 2 botella(s) en p6. 


--- ESTADO FINAL ---
1 (robot (posicion almacen) (botellas 3) (acciones 39))
2 (capacidad 3)
3 (siguiente almacen p1)
4 (siguiente p1 p2)
5 (siguiente p2 p3)
6 (siguiente p3 p4)
7 (siguiente p4 p5)
8 (siguiente p5 p6)
9 (siguiente p6 almacen)
16 (servido p1 1)
17 (servido p3 2)
18 (servido p2 3)
19 (servido p4 1)
20 (servido p6 2)
21 (servido p5 3)


## Conclusiones

- **Base de hechos:** una plantilla `robot` (posición, botellas, acciones), hechos `siguiente` para los arcos dirigidos de la vía, `pedido` y `servido` para las solicitudes, y `capacidad` como hecho de control.
- **Reglas:** `cargar`, `servir`, `mover` y `volver-almacen` modelan las acciones del robot. La regla `objetivo` (prioridad 100) detecta el estado final y detiene el motor con `(halt)`.
- **Estrategia de control:** las prioridades hacen que el robot sea *voraz*: sirve en cuanto puede y recarga siempre que pasa por el almacén. Si no lleva botellas suficientes para un puesto, sigue de largo y lo atiende en la vuelta siguiente, respetando que cada pedido se entrega completo de una vez.
- **Verificación:** cada regla se ha probado por separado con un estado preparado (Paso 3), comprobando el estado resultante con `assert`. La ejecución completa (Paso 4) alcanza el estado objetivo en el escenario de prueba y en un escenario adicional con pedidos en los 6 puestos.
- **Posible mejora:** la estrategia no garantiza el plan con menos movimientos. Por ejemplo, el robot sale del almacén con 3 botellas aunque el siguiente pedido que puede servir esté lejos. Para buscar el plan óptimo haría falta una búsqueda en el espacio de estados, que no es el objetivo de un sistema de reglas hacia delante como este.

In [15]:
env.clear()
print("Entorno de CLIPS limpiado correctamente.")

Entorno de CLIPS limpiado correctamente.
